1- Training models

In [0]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    KFold
)

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)

2- Load data

In [0]:
df = pd.read_csv("bandgap_feature_matrix.csv")

print(df.shape)

3- Define x and y

In [0]:
target = "bandgap_energy"

X = df.drop(columns=[target])
X = X.fillna(X.median())
y = df[target]

4- Train/Test Split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

5- Random Forest

In [0]:
rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

print("Random Forest")
print("R² :", r2_score(y_test, pred_rf))
print("MAE:", mean_absolute_error(y_test, pred_rf))
print("RMSE:", root_mean_squared_error(y_test, pred_rf))

6- Gradient Boosting

In [0]:
gb = GradientBoostingRegressor(
    random_state=42
)

gb.fit(X_train, y_train)

pred_gb = gb.predict(X_test)

print("\nGradient Boosting")
print("R² :", r2_score(y_test, pred_gb))
print("MAE:", mean_absolute_error(y_test, pred_gb))
print("RMSE:", root_mean_squared_error(y_test, pred_gb))

7- Cross Validation

In [0]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rf,
    X,
    y,
    cv=cv,
    scoring="r2"
)

print("\nRandom Forest CV")
print("Mean R²:", scores.mean())
print("Std R² :", scores.std())

8- Feature Importance

In [0]:
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

display(importance_df.head(20))

Scientific takeaway

The bandgap model achieved an R² of approximately 0.70 on the hold-out test set and a cross-validated R² of approximately 0.63. Structural descriptors, particularly lattice-related features, tolerance factor, and octahedral factor, emerged among the most influential predictors, highlighting the importance of crystal geometry in governing the electronic structure of perovskite photocatalysts.

# MODEL INTERPRETATION


1- Parity plot

In [0]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error

# prediction error
errors = pred_gb - y_test

plt.figure(figsize=(7,7))

sc = plt.scatter(
    y_test,
    pred_gb,
    c=errors,
    cmap="coolwarm",
    s=80,
    alpha=0.8
)

# ideal prediction line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "--",
    linewidth=2,
    label="Perfect Prediction"
)

# metrics box
plt.text(
    0.05,
    0.95,
    (
        f"R² = {r2_score(y_test, pred_gb):.3f}\n"
        f"MAE = {mean_absolute_error(y_test, pred_gb):.3f} eV"
    ),
    transform=plt.gca().transAxes,
    verticalalignment="top",
    bbox=dict(
        boxstyle="round",
        facecolor="white",
        alpha=0.9
    )
)

# colorbar
cbar = plt.colorbar(sc)
cbar.set_label("Prediction Error (eV)")

plt.xlabel("Experimental Bandgap (eV)", fontsize=12)
plt.ylabel("Predicted Bandgap (eV)", fontsize=12)

plt.title(
    "Bandgap Prediction Performance\n"
    "Red = Overprediction | Blue = Underprediction",
    fontsize=13
)

plt.legend()

plt.tight_layout()

plt.show()

2- Save Feature Importance _Figure_

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from sklearn.metrics import r2_score

# ----------------------------------
# FEATURE IMPORTANCE
# ----------------------------------

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": gb.feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

top20 = importance_df.head(20)

# ----------------------------------
# COLORS
# ----------------------------------

norm = Normalize(
    vmin=top20["importance"].min(),
    vmax=top20["importance"].max()
)

colors = plt.cm.coolwarm(
    norm(top20["importance"][::-1])
)

# ----------------------------------
# PLOT
# ----------------------------------

plt.figure(figsize=(10,7))

bars = plt.barh(
    top20["feature"][::-1],
    top20["importance"][::-1],
    color=colors
)

# value labels
for bar in bars:

    width = bar.get_width()

    plt.text(
        width + 0.002,
        bar.get_y() + bar.get_height()/2,
        f"{width:.3f}",
        va="center",
        fontsize=9
    )

# optional threshold line
plt.axvline(
    x=0.05,
    linestyle="--",
    color="gray",
    alpha=0.4
)

plt.xlabel(
    "Feature Importance",
    fontsize=12
)

plt.ylabel(
    "Feature",
    fontsize=12
)

plt.title(
    f"Top 20 Features for Bandgap Prediction\n"
    f"Gradient Boosting (R² = {r2_score(y_test, pred_gb):.3f})",
    fontsize=14
)

plt.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "bandgap_feature_importance.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

3- SHAP analysis

SHAP (SHapley Additive exPlanations) is a game theoretic approach used to explain the output of any machine learning model. It breaks down a model's prediction into individual contributions from each input feature, making complex "black box" models transparent and interpretable.

In [0]:
%pip install shap

In [0]:
import shap

explainer = shap.TreeExplainer(gb)

shap_values = explainer.shap_values(X_test)

In [0]:
shap.summary_plot(
    shap_values,
    X_test,
    max_display=15,
    show=False
)

plt.tight_layout()

4- Save Best Model

In [0]:
import joblib

joblib.dump(
    gb,
    "bandgap_model.pkl"
)

print("Model saved.")

5- Save Predictions

In [0]:
pred_df = pd.DataFrame({
    "experimental_bandgap": y_test,
    "predicted_bandgap": pred_gb
})

pred_df.to_csv(
    "bandgap_predictions.csv",
    index=False
)

pred_df.head()

6. Scientific conclusion from the bandgap model

The bandgap model achieved a test-set R² of 0.70 and a cross-validated R² of 0.63. SHAP analysis revealed that lattice-related descriptors, electron affinity, and perovskite structural factors were among the most influential predictors. The importance of lattice descriptors and octahedral/tolerance factors indicates that crystal geometry plays a major role in determining the electronic structure of the studied perovskite photocatalysts.